# Load libraries

In [ ]:
from zzzs_util import *

import os
import gc
import ctypes
import joblib
from tqdm.auto import tqdm

libc = ctypes.CDLL("libc.so.6")

import pytz
import cudf
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

from sklearn.preprocessing import RobustScaler

from scipy.ndimage import gaussian_filter1d

In [ ]:
print(cudf.__version__)

In [ ]:
COMP_LOC = "/kaggle/input/child-mind-institute-detect-sleep-states/"
DATA_LOC = "/kaggle/input/zzzs-seriesfolds-test/"

NUM_FOLDS = 7

# Custom functions

In [ ]:
def data_cleaning(data):
    
    # Data cleaning
    data = data.fillna(0)
    data = data.astype(float)
    data = data.to_numpy()
    
    return data

In [ ]:
def feature_engineering_small(data):
    
    import cudf
    import pandas as pd
    import numpy as np
    
    # Time-based Features
    data['timestamp'] = cudf.to_datetime(data['timestamp'])
    data['weekday'] = data['timestamp'].dt.weekday
    data['is_weekend'] = (data['weekday'] >= 5).astype(int)
    data['hour'] = data['timestamp'].dt.hour
    data['day_of_week'] = data['timestamp'].dt.dayofweek
    data['day_of_month'] = data['timestamp'].dt.day

    # The step number per 24h period can be derived from the step column assuming each step is 5 seconds apart
    data['step_24h'] = (data['step'] % (24*60*12))

    # Statistical Features
#     data['enmo_mean'] = data['enmo'].mean()
#     data['enmo_var'] = data['enmo'].var()
#     data['anglez_mean'] = data['anglez'].mean()
#     data['anglez_var'] = data['anglez'].var()

#     # Rolling Aggregates
#     window_sizes = [int(x*12) for x in [5/60, 1, 2, 3, 4, 5, 6, 7, 8]]  # Convert hours to steps (5 seconds per step)
#     for window in window_sizes:
#         data[f'enmo_rolling_mean_{window}'] = data['enmo'].rolling(window=window).mean()
#         data[f'enmo_rolling_max_{window}'] = data['enmo'].rolling(window=window).max()
#         data[f'enmo_rolling_std_{window}'] = data['enmo'].rolling(window=window).std()

#         data[f'anglez_rolling_mean_{window}'] = data['anglez'].rolling(window=window).mean()
#         data[f'anglez_rolling_max_{window}'] = data['anglez'].rolling(window=window).max()
#         data[f'anglez_rolling_std_{window}'] = data['anglez'].rolling(window=window).std()

#         # For total variation (or first variation)
#         data[f'enmo_1v_rolling_mean_{window}'] = data['enmo'].diff().rolling(window=window).mean()
#         data[f'enmo_1v_rolling_max_{window}'] = data['enmo'].diff().rolling(window=window).max()
#         data[f'enmo_1v_rolling_std_{window}'] = data['enmo'].diff().rolling(window=window).std()

#         data[f'anglez_1v_rolling_mean_{window}'] = data['anglez'].diff().rolling(window=window).mean()
#         data[f'anglez_1v_rolling_max_{window}'] = data['anglez'].diff().rolling(window=window).max()
#         data[f'anglez_1v_rolling_std_{window}'] = data['anglez'].diff().rolling(window=window).std()
    
    return data

In [ ]:
# def generate_target_array(data_length, targets, sigma=10):
#     """
#     Generates a Gaussian heatmap binary target numpy array based on the provided targets.
    
#     Parameters:
#     - data_length (int): Length of the data series.
#     - targets (list of tuples): Each tuple is a pair of start and end indices for sleep events.
#     - sigma (float): Standard deviation for the Gaussian.
    
#     Returns:
#     - numpy array: Heatmap target array where the first column is for sleep onset and
#                     the second column is for wake times.
#     """
#     y = np.zeros((data_length, 2))  # Initialize the target array

#     # Generate Gaussians for each target range
#     for start, end in targets:
#         # Set Gaussian at the start of the sleep event
#         y[int(start), 0] = 1
#         # Set Gaussian at the end of the sleep event
#         y[int(end), 1] = 1
    
#     # Apply Gaussian filter to create the heatmap
#     y[:, 0] = gaussian_filter1d(y[:, 0], sigma=sigma, mode='wrap')
#     y[:, 1] = gaussian_filter1d(y[:, 1], sigma=sigma, mode='wrap')

#     # Normalize the heatmaps
#     y_max = y.max()
#     if y_max > 0:
#         y /= y_max

#     return y

In [ ]:
from math import exp, sqrt, pi

def gauss(n, sigma):
    """
    Gaussian distribution function.

    Parameters:
    - n (int): The number of points in the Gaussian distribution.
    - sigma (float): The standard deviation of the Gaussian distribution.

    Returns:
    - list: Values of the Gaussian distribution.
    """
    r = range(-int(n/2), int(n/2)+1)
    return [1 / (sigma * sqrt(2*pi)) * exp(-float(x)**2 / (2*sigma**2)) for x in r]

def generate_target_array(data_length, targets, sigma=240):
    """
    Generates a Gaussian heatmap binary target numpy array based on the provided targets.

    Parameters:
    - data_length (int): Length of the data series.
    - targets (list of tuples): Each tuple is a pair of start and end indices for sleep events.
    - sigma (float): Standard deviation for the Gaussian.

    Returns:
    - numpy array: Heatmap target array where the first column is for sleep onset and
                    the second column is for wake times.
    """
    target_gaussian = np.zeros((data_length, 2))
    gaussian_values = gauss(n=sigma*3, sigma=sigma)  # Generate Gaussian values

    for start, end in targets:
        st1, st2 = max(0, int(start) - len(gaussian_values)//2), int(start) + len(gaussian_values)//2 + 1
        ed1, ed2 = int(end) - len(gaussian_values)//2, min(data_length, int(end) + len(gaussian_values)//2 + 1)

        # Apply Gaussian values to the start and end indices, handling boundaries
        target_gaussian[st1:st2, 0] += gaussian_values[st1 - (int(start) - len(gaussian_values)//2):]
        target_gaussian[ed1:ed2, 1] += gaussian_values[:len(gaussian_values) - ((int(end) + len(gaussian_values)//2 + 1) - ed2)]

    # Normalize the heatmaps
    y_max = target_gaussian.max(axis=0)
    target_gaussian[:, 0] /= y_max[0] if y_max[0] > 0 else 1
    target_gaussian[:, 1] /= y_max[1] if y_max[1] > 0 else 1

    return target_gaussian

In [ ]:
# def generate_target_array(data_length, targets):
#     """
#     Generates a binary target numpy array based on the provided targets.
    
#     Parameters:
#     - data_length (int): Length of the data series.
#     - targets (list of tuples): Each tuple is a pair of start and end indices for sleep events.
    
#     Returns:
#     - numpy array: Binary target array where 1 indicates awake and 0 indicates sleep.
#     """
#     y = np.ones(data_length, dtype=int)  # Start with an array of ones (awake)

#     for start, end in targets:
#         y[int(start):int(end)+1] = 0  # Set the indices corresponding to sleep events to 0

#     return y.reshape(-1, 1)

# Create [fold]ers

In [ ]:
# Define the base path where you want to create the folders
base_path = './'

# List of folder names
folder_names = [f'fold{i}' for i in range(1,NUM_FOLDS+1)]

# Loop through the list and create each folder
for folder in folder_names:
    folder_path = os.path.join(base_path, folder)
    
    # Check if the folder already exists to avoid throwing an error
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# Events data

In [ ]:
PATH = COMP_LOC + "train_events.csv"

events_df = pd.read_csv(PATH)

events_df = events_df[events_df['step'].notnull()]
events_df.drop_duplicates(inplace=True)
events_df = events_df.reset_index(drop=True)

print(events_df.shape)
events_df.head()

# NumpyFolds

In [ ]:
# foldrun = 1

for fold in tqdm(range(1,NUM_FOLDS+1)):
    
#     if fold != foldrun: continue
    
    FOLD_PATH = DATA_LOC + f"fold{fold}/"
    
    for file in tqdm(os.listdir(FOLD_PATH)):
        
        
        ############
        # SERIES ID
        ############
        
        series_id = file.split('.parquet')[0]
        TRUNC_SERIES = False
        
        # Load the data with cuDF
        PATH = FOLD_PATH + f"{series_id}.parquet"
        data = cudf.read_parquet(PATH)
        data = data.reset_index(drop=True)
        
        ############
        # Target
        ############
        
        targets = []
        events = events_df[events_df.series_id == series_id]
        
#         events['night_diff'] = events['night'].diff()
#         if events['night_diff'].max() > 1:
#             TRUNC_SERIES = True
#         events.drop(columns=['night_diff'], inplace=True)
        
            
        trunc_data = pd.DataFrame()
        for i in range(len(events)-1):
            if events.iloc[i].event =='onset' and events.iloc[i+1].event =='wakeup' and events.iloc[i].night==events.iloc[i+1].night:
                start,end = events.timestamp.iloc[i],events.timestamp.iloc[i+1]
                
#                 if TRUNC_SERIES:
#                     # Insert logic to only day, day+1 from series where start occurs
#                     pass
                    

                start_id = data.loc[data.timestamp ==start].index.values[0]
                end_id = data.loc[data.timestamp ==end].index.values[0]
                targets.append((start_id,end_id))
                
        
        ############
        # Features
        ############
        
        # Feature Engineering
        data = feature_engineering_small(data)
        
        free_memory()
        
        # Scaling
#         features_to_scale = [column for column in data.columns if column not in ['series_id', 'step', 'timestamp']]
#         scaler = RobustScaler()
#         data[features_to_scale] = scaler.fit_transform(data[features_to_scale].to_pandas())
        
        
        # Final features
        data.drop(columns=['series_id','timestamp'], inplace=True)
#         np.save(f"./fold{fold}/{series_id}.npy", data.to_pandas().values)

        data = data.to_pandas()
        free_memory()

        data = data_cleaning(data)
        free_memory()
        
        # Assuming the original scaler was fitted with 121 features
        original_feature_count = 121

        # Create a dummy array with zeros where the rest of the features would be
        # This assumes that the scaler was fitted with zeros for missing features
        # If the scaler was fitted with NaNs for missing features, use np.nan instead of 0
        dummy_data = np.zeros((data.shape[0], original_feature_count))
        # Replace the first n columns of the dummy data with your actual data
        # Where 'n' is the number of features in 'data' which is less than 121
        dummy_data[:, :data.shape[1]] = data

#         print("Data cleaned")

        y = generate_target_array(data.shape[0], targets)
        free_memory()

#         print("Target data generated")
        
        scaler = joblib.load(f"/kaggle/input/zzzs-stdscalerfit/scaler_fold{fold}.pkl")

        # Scale the data
        X = scaler.transform(dummy_data)
        free_memory()

#         print("Scaler transform complete")


#         joblib.dump((targets, data, series_id), f"./fold{fold}/{series_id}.pkl")
        
        joblib.dump((X[:, :data.shape[1]], y, series_id), f"./fold{fold}/{series_id}.pkl")
        

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



sns.displot(y)

plt.ylim(0,2000)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



sns.displot(y)

# plt.ylim(0,100)

# References

In [ ]:
# https://www.kaggle.com/code/werus23/sleep-critical-point-prepare-data